# Exercises - Lesson 1.3: Math Foundations 3 - Transformer Architecture

In [ ]:
!pip install -q numpy torch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

## Skill 1 - Matmul at scale

In [ ]:
A = torch.tensor([[1.,2.,1.,0.], [0.,1.,2.,3.], [2.,0.,1.,1.]])
B = torch.tensor([[1.,2.], [0.,1.], [3.,0.], [1.,1.]])
C = A @ B
print(C)

## Skill 2 - Scaled dot-product attention

In [ ]:
def attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    return weights @ V

torch.manual_seed(0)
Q = torch.randn(3, 4)
K = torch.randn(3, 4)
V = torch.randn(3, 4)
out = attention(Q, K, V)
print(f'output shape: {out.shape}')
weights = torch.softmax(Q @ K.T / math.sqrt(4), dim=-1)
print(f'attn weights row 0: {weights[0].round(decimals=3)}')

## Skill 3 - Activation functions

In [ ]:
x = torch.tensor([-2.0, -0.5, 0.0, 0.5, 2.0])
print(f'input: {x.tolist()}')
print(f'ReLU:  {F.relu(x).tolist()}')
print(f'GELU:  {[round(v, 3) for v in F.gelu(x).tolist()]}')
print(f'SiLU:  {[round(v, 3) for v in F.silu(x).tolist()]}')

## Bonus - Linear collapse (why activations matter)

In [ ]:
import numpy as np
W1 = np.array([[1, 0], [1, 1]])
W2 = np.array([[2, 1], [0, 1]])
x  = np.array([1, 2])

# Two linear layers collapse to one matrix
print('two layers:', W2 @ (W1 @ x))     # [5 3]
print('combined:  ', (W2 @ W1) @ x)     # [5 3]  - identical

# A ReLU between them breaks the collapse
x2 = np.array([1, -3])
h  = np.maximum(0, W1 @ x2)
print('with ReLU: ', W2 @ h)            # [2 0]
print('collapsed: ', (W2 @ W1) @ x2)    # [0 -2]  - no single matrix matches

## Full transformer attention block

In [ ]:
class AttentionBlock(nn.Module):
    def __init__(self, d_model=128, n_heads=8):
        super().__init__()
        self.d_model = d_model; self.n_heads = n_heads; self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model)
        )
    def forward(self, x):
        B, T, D = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        attn_out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, D)
        return self.ffn(self.out(attn_out))

block = AttentionBlock(d_model=128, n_heads=8)
x = torch.randn(2, 16, 128)
out = block(x)
print(f'output shape: {out.shape}')